# 🔬 Model Evaluation: Qwen2.5-7B Itemset Extractor v2

**Self-contained evaluation notebook for TLJH server.**

Evaluates the SFT-trained model on N random datasets vs Apriori ground truth.

**Metrics:** Precision, Recall, F1, Exact Match, JSON Parse Rate, Hallucination Rate, Think Rate, Count Accuracy, Inference Time

## Setup
1. Upload `datasets_v2.zip` to the same directory as this notebook
2. Run all cells
3. Results saved to `eval_results/`

## ⚡ Adapter-Only Mode (v2.1 — Council Diagnostic)
Set `"load_mode": "adapters"` in Cell 2 to load LoRA adapters on the base model
**without** the destructive `merged_4bit_forced` merge. This is the critical diagnostic
to confirm whether the merge was destroying fine-tuned behavior.

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth pandas

In [ ]:
# Cell 2: Configuration — EDIT THIS CELL
CONFIG = {
    # ── Model loading ──
    # "merged"   → load merged model from HuggingFace (v2 default — BROKEN by merged_4bit_forced)
    # "adapters" → load base model + LoRA adapters separately (v2.1 diagnostic — RECOMMENDED)
    "load_mode": "adapters",
    
    # For load_mode="merged":
    "model_name": "OliverSlivka/qwen2.5-7b-itemset-extractor",
    
    # For load_mode="adapters":
    "base_model": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    "adapter_path": "./sft_checkpoint",   # Path to saved LoRA adapters from training Cell 9
    
    "max_seq_length": 4096,
    
    # Evaluation
    "dataset_dir": "./datasets_v2",      # Directory with CSV files
    "num_datasets": 50,                   # How many random datasets to evaluate
    "min_support": 3,                     # Apriori min support
    "max_size": 3,                        # Max itemset size (singles, pairs, triples)
    "max_new_tokens": 4096,               # Generation length (was 2048, increased after diagnostic)
    "temperature": 0.1,                   # Low = deterministic
    "seed": 42,                           # Reproducible sampling
    
    # Output
    "output_dir": "./eval_results",
    
    # Targets
    "target_f1": 0.80,
    "target_parse_rate": 0.90,
    "target_max_hallucination": 0.05,
}
print("✅ Config ready")
print(f"   Load mode:  {CONFIG['load_mode']}")
if CONFIG["load_mode"] == "adapters":
    print(f"   Base model: {CONFIG['base_model']}")
    print(f"   Adapters:   {CONFIG['adapter_path']}")
else:
    print(f"   Model: {CONFIG['model_name']}")
print(f"   Datasets: {CONFIG['num_datasets']} from {CONFIG['dataset_dir']}")

In [ ]:
# Cell 3: Unzip datasets (run once)
import os, zipfile, glob

dataset_dir = CONFIG["dataset_dir"]
zip_path = "./datasets_v2.zip"

# Check if already unzipped
existing = glob.glob(os.path.join(dataset_dir, "*.csv"))
if len(existing) > 100:
    print(f"✅ Already have {len(existing)} CSVs in {dataset_dir}")
elif os.path.exists(zip_path):
    print(f"📦 Unzipping {zip_path}...")
    os.makedirs(dataset_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dataset_dir)
    existing = glob.glob(os.path.join(dataset_dir, "*.csv"))
    # Handle nested directory from zip
    if not existing:
        # Check if zip created a subdirectory
        for root, dirs, files in os.walk(dataset_dir):
            csvs = [f for f in files if f.endswith('.csv')]
            if csvs:
                print(f"   Found CSVs in {root}, moving to {dataset_dir}")
                for f in csvs:
                    src = os.path.join(root, f)
                    dst = os.path.join(dataset_dir, f)
                    if src != dst:
                        os.rename(src, dst)
                break
        existing = glob.glob(os.path.join(dataset_dir, "*.csv"))
    print(f"✅ Unzipped {len(existing)} CSV files")
else:
    print(f"❌ No datasets found!")
    print(f"   Upload datasets_v2.zip to this directory, OR")
    print(f"   Place CSV files directly in {dataset_dir}/")

In [ ]:
# Cell 4: GPU check + imports (nvidia-smi first to avoid CUDA OOM on busy GPU 0)
import subprocess, os, json, re, time, random, itertools, gc
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict

# ── Suppress TF/JAX (TLJH has Keras 3 which crashes transformers) ──
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# ── Find free GPU via nvidia-smi BEFORE importing torch ──
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.free,memory.total,name",
         "--format=csv,noheader,nounits"],
        text=True
    )
    gpus = []
    for line in smi.strip().split("\n"):
        parts = [p.strip() for p in line.split(",")]
        gpus.append({"idx": int(parts[0]), "free_mb": int(parts[1]),
                      "total_mb": int(parts[2]), "name": parts[3]})
    
    best = max(gpus, key=lambda g: g["free_mb"])
    print(f"🖥️ {len(gpus)} GPU(s) detected via nvidia-smi:")
    for g in gpus:
        marker = " ← selected" if g["idx"] == best["idx"] else ""
        print(f"   GPU {g['idx']}: {g['name']} — {g['total_mb']/1024:.1f} GB total, {g['free_mb']/1024:.1f} GB free{marker}")
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(best["idx"])
    print(f"\n   ✅ CUDA_VISIBLE_DEVICES={best['idx']} ({best['free_mb']/1024:.1f} GB free)")
except Exception as e:
    print(f"⚠️ nvidia-smi failed ({e}), using default GPU")

import torch
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    print(f"   torch sees GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️ No GPU — inference will be very slow")

In [ ]:
# Cell 5: Load model
from unsloth import FastLanguageModel

if CONFIG["load_mode"] == "adapters":
    # ═══════════════════════════════════════════════════════════════════════
    # ADAPTER MODE: Unsloth natively loads base + LoRA from adapter dir
    # Point from_pretrained at the adapter checkpoint — Unsloth auto-detects
    # the base model from adapter_config.json and loads both correctly.
    # This keeps everything in Unsloth's optimized inference path.
    # ═══════════════════════════════════════════════════════════════════════
    print(f"📥 Loading adapters via Unsloth: {CONFIG['adapter_path']}")
    print(f"   (Unsloth reads base model from adapter_config.json automatically)")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CONFIG["adapter_path"],
        max_seq_length=CONFIG["max_seq_length"],
        load_in_4bit=True,
        dtype=None,
    )
    FastLanguageModel.for_inference(model)
    print(f"✅ Model ready (base + LoRA adapters via Unsloth, NO merge)")

else:
    # ═══════════════════════════════════════════════════════════════════════
    # MERGED MODE: Load pre-merged model from HuggingFace
    # ⚠️ WARNING: merged_4bit_forced destroys fine-tuned behavior!
    # ═══════════════════════════════════════════════════════════════════════
    print(f"📥 Loading MERGED model: {CONFIG['model_name']}")
    print("⚠️  WARNING: merged_4bit_forced may have destroyed LoRA deltas!")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CONFIG["model_name"],
        max_seq_length=CONFIG["max_seq_length"],
        load_in_4bit=True,
        dtype=None,
    )
    FastLanguageModel.for_inference(model)
    print(f"✅ Model loaded (merged 4-bit)")

if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name()}")
    print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Cell 6: System prompt + helper functions
# MUST match training_utils.py EXACTLY
SYSTEM_PROMPT = (
    "You are a frequent itemset extractor. Given CSV transaction data and a "
    "minimum support count, identify all itemsets whose items co-occur in at "
    "least that many rows.\n\n"
    "Rules:\n"
    "1. Scan single items, pairs, and triples (up to size 3)\n"
    "2. Count = number of distinct rows containing ALL items in the itemset\n"
    "3. Only report itemsets with count >= min_support\n"
    "4. Canonicalize items: lowercase, trimmed, sorted alphabetically\n"
    '5. Row references: \"Row N\" format, 1-based indexing\n\n'
    "Think step by step inside <think> tags, then output ONLY a JSON array:\n"
    '[{\"itemset\": [\"item1\", \"item2\"], \"count\": N, \"rows\": [\"Row 1\", \"Row 3\"]}]'
)


def load_csv(csv_path):
    """Load CSV → (transactions, formatted_text, n_rows, n_cols, all_items)"""
    df = pd.read_csv(csv_path)
    n_rows, n_cols = df.shape
    transactions, rows_text, all_items = [], [], set()
    for idx, row in df.iterrows():
        items = [str(v).strip() for v in row.values if pd.notna(v) and str(v).strip()]
        transactions.append(items)
        rows_text.append(f"Row {idx + 1}: {', '.join(items)}")
        for item in items:
            all_items.add(item.lower())
    return transactions, "\n".join(rows_text), n_rows, n_cols, all_items


def apriori(transactions, min_support=3, max_size=3):
    """Apriori algorithm — deterministic ground truth."""
    if not transactions:
        return []
    row_labels = [f"Row {i+1}" for i in range(len(transactions))]
    counts = {}
    # Level 1
    for idx, trans in enumerate(transactions):
        seen = set()
        for item in trans:
            norm = str(item).strip().lower()
            if not norm or norm in seen:
                continue
            seen.add(norm)
            k = (norm,)
            if k not in counts:
                counts[k] = {"count": 0, "rows": []}
            counts[k]["count"] += 1
            counts[k]["rows"].append(row_labels[idx])
    
    def prune(d):
        return {k: v for k, v in d.items() if v["count"] >= min_support}
    
    L1 = prune(counts)
    if not L1:
        return []
    
    freq_levels = [L1]
    current = L1
    k = 2
    while k <= max_size and current:
        prev_keys = sorted(current.keys())
        candidates = set()
        for i in range(len(prev_keys)):
            for j in range(i+1, len(prev_keys)):
                a, b = prev_keys[i], prev_keys[j]
                if a[:k-2] == b[:k-2]:
                    merged = tuple(sorted(set(a) | set(b)))
                    if len(merged) == k:
                        if all(tuple(sorted(sub)) in current
                               for sub in itertools.combinations(merged, k-1)):
                            candidates.add(merged)
        if not candidates:
            break
        cand_counts = {c: {"count": 0, "rows": []} for c in candidates}
        for idx, trans in enumerate(transactions):
            tset = {str(x).strip().lower() for x in trans}
            for cand in candidates:
                if set(cand).issubset(tset):
                    cand_counts[cand]["count"] += 1
                    cand_counts[cand]["rows"].append(row_labels[idx])
        current = prune(cand_counts)
        if current:
            freq_levels.append(current)
        k += 1
    
    out = []
    for level in freq_levels:
        for itemset, info in level.items():
            out.append({"itemset": list(itemset), "count": info["count"], "rows": info["rows"]})
    return out


def run_inference(csv_text, min_support):
    """Run model inference — returns dict with parsed results."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Find all frequent itemsets with minimum support count = {min_support} in this dataset:\n\n{csv_text}"},
    ]
    tokenized = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    # Build proper attention_mask (all 1s — no padding in single-sequence inference)
    if isinstance(tokenized, dict):
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized.get("attention_mask", torch.ones_like(input_ids))
    else:
        input_ids = tokenized
        attention_mask = torch.ones_like(input_ids)

    if torch.cuda.is_available():
        input_ids = input_ids.to(model.device)
        attention_mask = attention_mask.to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=CONFIG["temperature"],
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    
    # Parse
    has_think = "<think>" in raw and "</think>" in raw
    think_content = ""
    json_text = raw
    if has_think:
        parts = raw.split("</think>", 1)
        think_content = parts[0].split("<think>", 1)[-1].strip()
        json_text = parts[1].strip() if len(parts) > 1 else ""
    
    parsed, parse_ok = None, False
    try:
        parsed = json.loads(json_text)
        parse_ok = True
    except:
        m = re.search(r"\[.*\]", json_text, re.DOTALL)
        if m:
            try:
                parsed = json.loads(m.group())
                parse_ok = True
            except:
                pass
    if not parse_ok:
        m = re.search(r"\[.*\]", raw, re.DOTALL)
        if m:
            try:
                parsed = json.loads(m.group())
                parse_ok = True
            except:
                pass
    if not parse_ok or not isinstance(parsed, list):
        parsed = []
        parse_ok = False
    
    # Normalize
    items = []
    for rec in (parsed or []):
        if not isinstance(rec, dict):
            continue
        itemset = rec.get("itemset", [])
        if not isinstance(itemset, list) or not itemset:
            continue
        count = rec.get("count", 0)
        rows = rec.get("rows", rec.get("row", rec.get("evidence_rows", [])))
        norm_itemset = sorted(str(x).strip().lower() for x in itemset)
        norm_rows = []
        for r in rows:
            if isinstance(r, int):
                norm_rows.append(f"Row {r}")
            elif isinstance(r, str) and re.match(r"Row \d+", r):
                norm_rows.append(r)
            else:
                try: norm_rows.append(f"Row {int(r)}")
                except: pass
        items.append({"itemset": norm_itemset, "count": count if isinstance(count, int) else 0, "rows": sorted(set(norm_rows))})
    
    return {"raw_output": raw, "parsed_items": items, "parse_ok": parse_ok, "has_think": has_think, "think_content": think_content}


def canon(itemset):
    return frozenset(str(x).strip().lower() for x in itemset)


def compute_metrics(apriori_sets, model_sets, all_csv_items, count_tolerance=1):
    """Compute all evaluation metrics for one dataset."""
    apr_c = {canon(s["itemset"]) for s in apriori_sets}
    mod_c = {canon(s["itemset"]) for s in model_sets}
    tp = apr_c & mod_c
    fp = mod_c - apr_c
    fn = apr_c - mod_c
    p = len(tp) / len(mod_c) if mod_c else 0.0
    r = len(tp) / len(apr_c) if apr_c else 0.0
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0.0
    
    # Count accuracy
    apr_count_map = {canon(s["itemset"]): s["count"] for s in apriori_sets}
    ct_ok = ct_total = 0
    for s in model_sets:
        c = canon(s["itemset"])
        if c in apr_count_map:
            ct_total += 1
            if abs(s["count"] - apr_count_map[c]) <= count_tolerance:
                ct_ok += 1
    count_acc = ct_ok / ct_total if ct_total > 0 else 0.0
    
    # Hallucination
    hall = sum(1 for s in model_sets if any(i not in all_csv_items for i in s["itemset"]))
    hall_rate = hall / len(model_sets) if model_sets else 0.0
    
    # Per-size breakdown
    def by_size(sets_list):
        result = {}
        for s in sets_list:
            sz = len(s["itemset"])
            result.setdefault(sz, set()).add(canon(s["itemset"]))
        return result
    apr_sz, mod_sz = by_size(apriori_sets), by_size(model_sets)
    size_bd = {}
    for sz in sorted(set(apr_sz) | set(mod_sz)):
        a, m = apr_sz.get(sz, set()), mod_sz.get(sz, set())
        s_tp = len(a & m)
        s_p = s_tp/len(m) if m else 0.0
        s_r = s_tp/len(a) if a else 0.0
        s_f1 = 2*s_p*s_r/(s_p+s_r) if (s_p+s_r)>0 else 0.0
        size_bd[sz] = {"apriori": len(a), "model": len(m), "tp": s_tp, "p": round(s_p,4), "r": round(s_r,4), "f1": round(s_f1,4)}
    
    return {
        "apriori_count": len(apr_c), "model_count": len(mod_c),
        "tp": len(tp), "fp": len(fp), "fn": len(fn),
        "precision": round(p,4), "recall": round(r,4), "f1": round(f1,4),
        "exact_match": f1 == 1.0,
        "count_accuracy": round(count_acc,4),
        "hallucination_rate": round(hall_rate,4),
        "size_breakdown": size_bd,
    }

print("✅ All helper functions defined")
print(f"   System prompt: {len(SYSTEM_PROMPT)} chars")

In [ ]:
# Cell 7: Select evaluation datasets
import glob

all_csvs = sorted(glob.glob(os.path.join(CONFIG["dataset_dir"], "*.csv")))
print(f"📊 Found {len(all_csvs)} CSV files in {CONFIG['dataset_dir']}")

# Random sample
random.seed(CONFIG["seed"])
if len(all_csvs) > CONFIG["num_datasets"]:
    eval_csvs = random.sample(all_csvs, CONFIG["num_datasets"])
else:
    eval_csvs = all_csvs

print(f"   Selected {len(eval_csvs)} for evaluation")
# Show first 5
for f in eval_csvs[:5]:
    print(f"   → {os.path.basename(f)}")
if len(eval_csvs) > 5:
    print(f"   ... and {len(eval_csvs)-5} more")

In [ ]:
# Cell 8: 🚀 RUN EVALUATION (main loop)
print("═" * 70)
print("  ITEMSET EXTRACTION MODEL EVALUATION")
print("═" * 70)
model_display = f"{CONFIG['adapter_path']} on {CONFIG['base_model']}" if CONFIG["load_mode"] == "adapters" else CONFIG["model_name"]
print(f"  Model:       {model_display}")
print(f"  Load mode:   {CONFIG['load_mode']}")
print(f"  max_tokens:  {CONFIG['max_new_tokens']}")
print(f"  Datasets:    {len(eval_csvs)}")
print(f"  Min support: {CONFIG['min_support']}")
print(f"  Max size:    {CONFIG['max_size']}")
print("═" * 70)

all_results = []
total_start = time.perf_counter()

for i, csv_path in enumerate(eval_csvs, 1):
    ds_name = os.path.basename(csv_path)
    print(f"\n[{i}/{len(eval_csvs)}] {ds_name}")
    
    try:
        # Load CSV
        transactions, csv_text, n_rows, n_cols, all_items = load_csv(csv_path)
        print(f"   Loaded: {n_rows} rows × {n_cols} cols")
        
        # Apriori ground truth
        t0 = time.perf_counter()
        apriori_sets = apriori(transactions, CONFIG["min_support"], CONFIG["max_size"])
        apr_time = time.perf_counter() - t0
        print(f"   Apriori: {len(apriori_sets)} itemsets ({apr_time*1000:.0f}ms)")
        
        # Model inference
        t0 = time.perf_counter()
        inference = run_inference(csv_text, CONFIG["min_support"])
        model_time = time.perf_counter() - t0
        model_sets = inference["parsed_items"]
        print(f"   Model:   {len(model_sets)} itemsets ({model_time:.1f}s) | JSON={'✅' if inference['parse_ok'] else '❌'} | <think>={'✅' if inference['has_think'] else '❌'}")
        
        # Metrics
        metrics = compute_metrics(apriori_sets, model_sets, all_items)
        
        result = {
            "dataset": ds_name, "n_rows": n_rows, "n_cols": n_cols,
            "apriori_count": len(apriori_sets), "model_count": len(model_sets),
            "apriori_time_s": round(apr_time, 3), "model_time_s": round(model_time, 2),
            "parse_ok": inference["parse_ok"], "has_think": inference["has_think"],
            "metrics": metrics, "raw_output": inference["raw_output"],
            "think_content": inference["think_content"],
        }
        all_results.append(result)
        
        m = metrics
        print(f"   → P={m['precision']:.0%} R={m['recall']:.0%} F1={m['f1']:.0%} | halluc={m['hallucination_rate']:.0%}")
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        all_results.append({"dataset": ds_name, "error": str(e)})

total_time = time.perf_counter() - total_start
print(f"\n⏱️ Total evaluation time: {total_time:.0f}s ({total_time/60:.1f}min)")

In [ ]:
# Cell 9: Compute aggregate summary
successful = [r for r in all_results if "metrics" in r]
n = len(successful)

if not successful:
    print("❌ No successful evaluations!")
else:
    model_label = (
        f"{CONFIG['base_model']} + {CONFIG['adapter_path']}"
        if CONFIG["load_mode"] == "adapters"
        else CONFIG["model_name"]
    )
    summary = {
        "model": model_label,
        "load_mode": CONFIG["load_mode"],
        "total": len(eval_csvs),
        "successful": n,
        "failed": len(all_results) - n,
        "avg_precision": round(sum(r["metrics"]["precision"] for r in successful) / n, 4),
        "avg_recall": round(sum(r["metrics"]["recall"] for r in successful) / n, 4),
        "avg_f1": round(sum(r["metrics"]["f1"] for r in successful) / n, 4),
        "exact_match_count": sum(1 for r in successful if r["metrics"]["exact_match"]),
        "exact_match_rate": round(sum(1 for r in successful if r["metrics"]["exact_match"]) / n, 4),
        "parse_rate": round(sum(1 for r in successful if r["parse_ok"]) / n, 4),
        "think_rate": round(sum(1 for r in successful if r["has_think"]) / n, 4),
        "avg_hallucination": round(sum(r["metrics"]["hallucination_rate"] for r in successful) / n, 4),
        "avg_count_accuracy": round(sum(r["metrics"]["count_accuracy"] for r in successful) / n, 4),
        "avg_time_s": round(sum(r["model_time_s"] for r in successful) / n, 2),
        "total_time_s": round(total_time, 1),
        "min_support": CONFIG["min_support"],
        "max_size": CONFIG["max_size"],
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    
    # ── Print summary ──
    print("\n" + "═" * 70)
    print("  EVALUATION SUMMARY")
    print("═" * 70)
    print(f"  Load mode:         {CONFIG['load_mode'].upper()}")
    print(f"  Datasets:          {n}/{len(eval_csvs)} successful")
    print(f"  Avg Precision:     {summary['avg_precision']:.1%}")
    print(f"  Avg Recall:        {summary['avg_recall']:.1%}")
    print(f"  Avg F1 Score:      {summary['avg_f1']:.1%}")
    print(f"  Exact Match:       {summary['exact_match_count']}/{n} ({summary['exact_match_rate']:.1%})")
    print(f"  JSON Parse Rate:   {summary['parse_rate']:.1%}")
    print(f"  Think Rate:        {summary['think_rate']:.1%}")
    print(f"  Hallucination:     {summary['avg_hallucination']:.1%}")
    print(f"  Count Accuracy:    {summary['avg_count_accuracy']:.1%}")
    print(f"  Avg Inference:     {summary['avg_time_s']:.1f}s per dataset")
    print(f"  Total Time:        {total_time:.0f}s")
    
    # ── Pass/Fail verdict ──
    print("\n" + "─" * 70)
    passed = (
        summary["avg_f1"] >= CONFIG["target_f1"]
        and summary["parse_rate"] >= CONFIG["target_parse_rate"]
        and summary["avg_hallucination"] <= CONFIG["target_max_hallucination"]
    )
    if passed:
        print("  🎉 PASSED — Model meets production targets!")
    else:
        reasons = []
        if summary["avg_f1"] < CONFIG["target_f1"]:
            reasons.append(f"F1 {summary['avg_f1']:.1%} < {CONFIG['target_f1']:.0%}")
        if summary["parse_rate"] < CONFIG["target_parse_rate"]:
            reasons.append(f"Parse rate {summary['parse_rate']:.1%} < {CONFIG['target_parse_rate']:.0%}")
        if summary["avg_hallucination"] > CONFIG["target_max_hallucination"]:
            reasons.append(f"Hallucination {summary['avg_hallucination']:.1%} > {CONFIG['target_max_hallucination']:.0%}")
        print(f"  ⚠️  NOT YET — {', '.join(reasons)}")
    print("─" * 70)

In [ ]:
# Cell 10: Save results to files
output_dir = Path(CONFIG["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

# Summary JSON
(output_dir / "evaluation_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

# All results (without raw_output to keep size manageable)
detail_results = []
for r in all_results:
    detail = {k: v for k, v in r.items() if k not in ("raw_output", "think_content")}
    detail_results.append(detail)
(output_dir / "all_results.json").write_text(
    json.dumps(detail_results, indent=2, ensure_ascii=False), encoding="utf-8"
)

# Save raw outputs for debugging (1 file per dataset)
raw_dir = output_dir / "raw_outputs"
raw_dir.mkdir(exist_ok=True)
for r in all_results:
    if "raw_output" in r:
        (raw_dir / f"{r['dataset'].replace('.csv','')}.txt").write_text(
            r["raw_output"], encoding="utf-8"
        )

print(f"💾 Results saved to {output_dir}/")
print(f"   evaluation_summary.json  — aggregate metrics")
print(f"   all_results.json         — per-dataset details")
print(f"   raw_outputs/             — model raw text output per dataset")

In [ ]:
# Cell 11: Detailed failure analysis
if successful:
    parse_failures = [r for r in all_results if not r.get("parse_ok", True) and "metrics" not in r]
    low_f1 = [r for r in successful if r["metrics"]["f1"] < 0.5]
    hallucinating = [r for r in successful if r["metrics"]["hallucination_rate"] > 0.1]
    perfect = [r for r in successful if r["metrics"]["exact_match"]]
    no_think = [r for r in successful if not r["has_think"]]
    
    print("📊 FAILURE ANALYSIS")
    print("═" * 70)
    print(f"  Perfect F1=1.0:     {len(perfect)}/{n} ({len(perfect)/n:.0%})")
    print(f"  F1 ≥ 0.8:          {sum(1 for r in successful if r['metrics']['f1']>=0.8)}/{n}")
    print(f"  F1 ≥ 0.5:          {sum(1 for r in successful if r['metrics']['f1']>=0.5)}/{n}")
    print(f"  F1 < 0.5 (bad):    {len(low_f1)}/{n}")
    print(f"  Parse failures:    {len(parse_failures)}")
    print(f"  Hallucinating:     {len(hallucinating)}/{n}")
    print(f"  No <think>:        {len(no_think)}/{n}")
    
    # F1 distribution
    f1_vals = sorted([r["metrics"]["f1"] for r in successful])
    print(f"\n  F1 Distribution:")
    print(f"    Min:    {f1_vals[0]:.3f}")
    print(f"    Q1:     {f1_vals[len(f1_vals)//4]:.3f}")
    print(f"    Median: {f1_vals[len(f1_vals)//2]:.3f}")
    print(f"    Q3:     {f1_vals[3*len(f1_vals)//4]:.3f}")
    print(f"    Max:    {f1_vals[-1]:.3f}")
    
    # Per-size aggregation
    all_size_data = defaultdict(list)
    for r in successful:
        for sz, data in r["metrics"]["size_breakdown"].items():
            all_size_data[sz].append(data)
    
    if all_size_data:
        print(f"\n  Performance by Itemset Size:")
        print(f"    {'Size':<6} {'Avg Apr':>8} {'Avg Mod':>8} {'Avg P':>7} {'Avg R':>7} {'Avg F1':>7}")
        for sz in sorted(all_size_data):
            entries = all_size_data[sz]
            nn = len(entries)
            print(f"    {sz:<6} {sum(e['apriori'] for e in entries)/nn:>8.1f} {sum(e['model'] for e in entries)/nn:>8.1f} "
                  f"{sum(e['p'] for e in entries)/nn:>6.0%} {sum(e['r'] for e in entries)/nn:>6.0%} "
                  f"{sum(e['f1'] for e in entries)/nn:>6.0%}")
    
    # Show worst 5 datasets
    if low_f1:
        print(f"\n  📉 Worst {min(5,len(low_f1))} datasets (F1 < 0.5):")
        for r in sorted(low_f1, key=lambda x: x['metrics']['f1'])[:5]:
            m = r['metrics']
            print(f"    {r['dataset']}: F1={m['f1']:.0%} P={m['precision']:.0%} R={m['recall']:.0%} (Apr={m['apriori_count']}, Mod={m['model_count']})")
    
    # Show sample raw output (first result)
    if successful:
        print(f"\n  📝 Sample output ({successful[0]['dataset']}):")
        raw = successful[0].get("raw_output", "")
        print(f"    {raw[:500]}{'...' if len(raw)>500 else ''}")

In [ ]:
# Cell 12: Print results as a table for easy copy-paste
if successful:
    model_label = (
        f"{CONFIG['base_model']} + {CONFIG['adapter_path']}"
        if CONFIG["load_mode"] == "adapters"
        else CONFIG["model_name"]
    )
    print("\n📋 RESULTS TABLE (copy this for the training agent)")
    print("═" * 70)
    print(f"""```
Model: {model_label}
Load mode: {CONFIG['load_mode']}
Datasets evaluated: {n}/{len(eval_csvs)}
Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}

AGGREGATE METRICS:
  F1 Score:        {summary['avg_f1']:.4f} ({summary['avg_f1']:.1%})
  Precision:       {summary['avg_precision']:.4f} ({summary['avg_precision']:.1%})
  Recall:          {summary['avg_recall']:.4f} ({summary['avg_recall']:.1%})
  Exact Match:     {summary['exact_match_count']}/{n} ({summary['exact_match_rate']:.1%})
  JSON Parse Rate: {summary['parse_rate']:.4f} ({summary['parse_rate']:.1%})
  Think Rate:      {summary['think_rate']:.4f} ({summary['think_rate']:.1%})
  Hallucination:   {summary['avg_hallucination']:.4f} ({summary['avg_hallucination']:.1%})
  Count Accuracy:  {summary['avg_count_accuracy']:.4f} ({summary['avg_count_accuracy']:.1%})
  Avg Inference:   {summary['avg_time_s']:.1f}s
  Total Time:      {summary['total_time_s']:.0f}s

VERDICT: {'PASSED ✅' if passed else 'NOT YET ⚠️'}
```""")
    print("═" * 70)
    print("\n💡 Copy the above block and paste it when running /validate with the training agent")